# Ejercicio: Agentes con LangGraph

**Resuelve al menos uno de los dos ejercicios. Si acabas el primero, intenta el segundo.**

| Ejercicio | Caso de uso | Dificultad |
|-----------|-------------|------------|
| **1** | Agente de Soporte IT | ⭐⭐ Media |
| **2** | Agente Text-to-SQL | ⭐⭐⭐ Alta |

Los dos usan el mismo stack: `TypedDict` + `@tool` + `StateGraph` + edge condicional.

## Setup — ejecuta esto primero

In [ ]:
!pip install -q langchain langchain-openai langgraph

In [ ]:
import os
import logging

os.environ["OPENAI_API_KEY"] = "sk-..."  # tu clave de OpenAI
os.environ["LANGSMITH_TRACING"] = "false"
logging.getLogger("langsmith").setLevel(logging.CRITICAL)

from langchain_openai import ChatOpenAI
from langchain_core.tools import tool
from langchain_core.messages import HumanMessage, AIMessage
from langgraph.graph import StateGraph, END
from langgraph.graph.message import add_messages
from typing import TypedDict, Annotated

llm = ChatOpenAI(model="gpt-4o-mini", temperature=0)
print("Listo.")

---
# Ejercicio 1 — Agente de Soporte IT ⭐⭐

Construye un agente que recibe un ticket de soporte, lo clasifica y devuelve una solución.

```
  Ticket
    │
    ▼
  [clasificar]  ──► "software" ──► [buscar_solucion_software]
       │                                      │
       ├──────────► "red"      ──► [buscar_solucion_red]      ──► [responder] ──► END
       │                                      │
       └──────────► "hardware" ──► [buscar_solucion_hardware]
```

**Requisitos mínimos:**
- `TypedDict` con: `messages`, `categoria`, `solucion`
- Al menos **2 herramientas** con `@tool`
- Un **edge condicional** basado en la categoría
- El agente responde con la solución encontrada

### 1.1 — Estado

In [ ]:
class EstadoIT(TypedDict):
    messages:  Annotated[list, add_messages]
    categoria: str   # "software", "red" o "hardware"
    solucion:  str   # la solución encontrada

### 1.2 — Herramientas

Implementa al menos 2. Pueden devolver respuestas simuladas — no necesitas una base de conocimiento real.

In [ ]:
@tool
def buscar_solucion_software(problema: str) -> str:
    """Busca soluciones para problemas de software: aplicaciones, instalaciones, licencias, crashes."""
    raise NotImplementedError("Implementa esta función")


@tool
def buscar_solucion_red(problema: str) -> str:
    """Busca soluciones para problemas de red: conexión, VPN, WiFi, DNS."""
    raise NotImplementedError("Implementa esta función")


# Opcional
# @tool
# def buscar_solucion_hardware(problema: str) -> str:
#     """Busca soluciones para problemas de hardware: ordenador lento, ruidos, pantalla."""
#     raise NotImplementedError("Implementa esta función")

### 1.3 — Nodos

**Pista `clasificar`:** pide al LLM que devuelva solo una palabra: `software`, `red` o `hardware`.  
**Pista `buscar`:** las tools se invocan así: `buscar_solucion_software.invoke({"problema": ticket})`

In [ ]:
def clasificar(state: EstadoIT) -> dict:
    raise NotImplementedError

def buscar(state: EstadoIT) -> dict:
    raise NotImplementedError

def responder(state: EstadoIT) -> dict:
    raise NotImplementedError

def enrutar(state: EstadoIT) -> str:
    raise NotImplementedError

### 1.4 — Grafo

In [ ]:
# g = StateGraph(EstadoIT)
# g.add_node(...)
# g.set_entry_point(...)
# g.add_conditional_edges(...)
# g.add_edge(...)
# app_it = g.compile()

raise NotImplementedError("Construye el grafo")

### 1.5 — Prueba con estos tickets

In [ ]:
tickets = [
    "No puedo instalar el software de contabilidad, me da error de licencia.",
    "No tengo acceso a internet desde esta mañana, el WiFi no conecta.",
    "Mi ordenador hace ruido raro y va muy lento, creo que es el ventilador.",
]

for ticket in tickets:
    print(f"\n{'='*55}")
    print(f"TICKET: {ticket}")
    resultado = app_it.invoke({
        "messages":  [HumanMessage(content=ticket)],
        "categoria": "",
        "solucion":  "",
    })
    print(f"CATEGORÍA: {resultado['categoria']}")
    print(f"RESPUESTA: {resultado['messages'][-1].content}")

---
# Ejercicio 2 — Agente Text-to-SQL ⭐⭐⭐

Construye un agente que responde preguntas en lenguaje natural sobre una base de datos SQLite.  
El agente genera el SQL, lo ejecuta y luego interpreta el resultado.

```
  Pregunta
     │
     ▼
  [generar_sql]  ──► [ejecutar_sql]  ──► [interpretar]  ──► END
                           │
                      error SQL?
                           │
                           ▼
                    [responder_error]
```

**Requisitos mínimos:**
- `TypedDict` con: `messages`, `sql_generado`, `resultado_sql`, `hay_error`
- Al menos **1 herramienta** `@tool` para ejecutar SQL
- Un **edge condicional** que detecte si hubo error en la ejecución

### 2.1 — Prepara la base de datos

Usamos la misma base de datos del Lab 2 (productos Apple + ventas).

In [ ]:
preguntas = [
    "¿Cuántas unidades se han vendido en total de cada producto?",
    "¿Cuál es el producto más caro?",
    "¿Cuánto dinero se generó en ventas online vs tienda?",
    "¿Qué producto tiene más stock disponible?",
    "Dame el top 3 de productos por volumen de ventas en euros.",
    "¿Qué día se vendieron más unidades en total?",
]

for pregunta in preguntas:
    print(f"\n{'='*55}")
    print(f"PREGUNTA: {pregunta}")
    resultado = app_sql.invoke({
        "messages":      [HumanMessage(content=pregunta)],
        "sql_generado":  "",
        "resultado_sql": "",
        "hay_error":     False,
    })
    print(f"SQL:      {resultado['sql_generado']}")
    print(f"RESPUESTA: {resultado['messages'][-1].content}")

### 2.2 — Estado

In [ ]:
class EstadoSQL(TypedDict):
    messages:      Annotated[list, add_messages]
    sql_generado:  str    # el SQL que generó el LLM
    resultado_sql: str    # el resultado de ejecutar el SQL
    hay_error:     bool   # True si el SQL falló

### 2.3 — Herramienta

**Pista:** la tool recibe una query SQL como string, la ejecuta contra `conn` y devuelve el resultado como string.

In [ ]:
@tool
def ejecutar_sql(query: str) -> str:
    """Ejecuta una consulta SQL contra la base de datos y devuelve el resultado como texto."""
    raise NotImplementedError("Implementa esta función")
    # Pista:
    # try:
    #     df = pd.read_sql(query, conn)
    #     return df.to_string(index=False)
    # except Exception as e:
    #     return f"ERROR: {e}"

### 2.4 — Nodos

**Pista `generar_sql`:** incluye el `ESQUEMA` en el prompt para que el LLM sepa las tablas y columnas.  
**Pista `enrutar_sql`:** detecta si `state["resultado_sql"]` empieza con `"ERROR"`.

In [ ]:
import re

def generar_sql(state: EstadoSQL) -> dict:
    """Genera el SQL a partir de la pregunta en lenguaje natural."""
    # Pista: usa ESQUEMA en el prompt y pide solo el SQL sin explicaciones
    raise NotImplementedError

def ejecutar(state: EstadoSQL) -> dict:
    """Ejecuta el SQL generado y guarda el resultado en el estado."""
    raise NotImplementedError

def interpretar(state: EstadoSQL) -> dict:
    """Interpreta el resultado SQL y genera una respuesta en lenguaje natural."""
    raise NotImplementedError

def responder_error(state: EstadoSQL) -> dict:
    """Informa al usuario de que no se pudo ejecutar la consulta."""
    raise NotImplementedError

def enrutar_sql(state: EstadoSQL) -> str:
    """Si hay error va a responder_error, si no va a interpretar."""
    raise NotImplementedError

### 2.5 — Grafo

In [ ]:
# g2 = StateGraph(EstadoSQL)
# ...
# app_sql = g2.compile()

raise NotImplementedError("Construye el grafo")

### 2.6 — Prueba con estas preguntas

In [ ]:
preguntas = [
    "¿Cuántas unidades se han vendido en total de cada producto?",
    "¿Cuál es el producto más caro?",
    "¿Cuánto dinero se generó en ventas online vs tienda?",
]

for pregunta in preguntas:
    print(f"\n{'='*55}")
    print(f"PREGUNTA: {pregunta}")
    resultado = app_sql.invoke({
        "messages":      [HumanMessage(content=pregunta)],
        "sql_generado":  "",
        "resultado_sql": "",
        "hay_error":     False,
    })
    print(f"SQL:      {resultado['sql_generado']}")
    print(f"RESPUESTA: {resultado['messages'][-1].content}")

---
## Bonus (opcional para los dos)

Si acabas los dos ejercicios, prueba añadir **Human-in-the-Loop** a cualquiera de ellos:
- En el **soporte IT**: el agente propone la solución y espera que el técnico la apruebe antes de enviársela al usuario
- En el **Text-to-SQL**: el agente muestra el SQL generado y espera confirmación antes de ejecutarlo

Pista: `MemorySaver` + `interrupt_after` + `update_state` — lo vimos en la Sección 5 del lab.